## Exercise — Overfitting: when a model learns the noise

### Task 1 — Read the train–test gap

Three solubility models were each scored on a training set and a test set:

| Model | Training $R^2$ | Test $R^2$ |
|---|---|---|
| model A | 0.41 | 0.38 |
| model B | 0.97 | 0.58 |
| model C | 0.86 | 0.82 |

```py
model_names = ["model A", "model B", "model C"]
r2_train_scores = [0.41, 0.97, 0.86]
r2_test_scores = [0.38, 0.58, 0.82]
```

Use the diagnosis table from the video, with these thresholds:

- a training $R^2$ below 0.5 means **underfitting**
- otherwise, a gap (training minus test) larger than 0.1 means **overfitting**
- otherwise, the model is a **good fit**

Walk the three lists together with `zip`. For each model, print its name, its gap rounded to 2 decimals, and its diagnosis.


### Task 2 — Fit both models and score them side by side

Start from this setup. It is the video's simulated solubility data, drawn with a different seed, so you get a fresh batch of 30 compounds. The true relationship is still the straight line $\log S = 0.5 - 0.8 \cdot \log P$.

```py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

np.random.seed(7)
logp = np.linspace(0.5, 5.0, 30)
noise = np.random.normal(0, 0.4, 30)
logs = 0.5 - 0.8 * logp + noise
df = pd.DataFrame({"logp": logp, "logs": logs})
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def line(logp, a, b):
    return a * logp + b

def quad(logp, a, b, c):
    return a * logp ** 2 + b * logp + c
```

Then do the following:

1. Fit `line` and `quad` with `curve_fit` on the **training rows only**. Unpack the parameters into `a_line, b_line` and `a_quad, b_quad, c_quad`.
2. Score each model on both sets with `r_squared`. Store the scores as `r2_line_train`, `r2_line_test`, `r2_quad_train` and `r2_quad_test`.
3. Print one line per model showing its training and test $R^2$ to 3 decimals.
4. Print the gap, training minus test, for each model, rounded to 3 decimals.

Use your rule from Task 1 to diagnose each model.


### Task 3 — What is guaranteed, and what is not

Each line below prints `True` or `False`. Before running anything, write down which lines are **guaranteed** to print `True` on any dataset, and which **could go either way**. Then run them and check.

```py
print(quad(3.0, 0, a_line, b_line) == line(3.0, a_line, b_line))
print(r2_quad_train >= r2_line_train)
print(r2_quad_test >= r2_line_test)
```

Hint: the first line sets the squared coefficient of the quadratic to zero. Keep in mind that a cubic or a higher-order polynomial would push the same trade further: its training score would be higher still, and its test score would still carry no guarantee.


### Task 4 — Bias and variance as a repeat measurement

In lab terms, **bias** is systematic error, and **variance** is poor repeatability. Test the repeatability of each model by "measuring the same compounds again on another day":

1. Draw new assay noise for the same 30 logP values with `np.random.normal(0, 0.4, 30)`. Build a second table, `batch2`, with columns `logp` and `logs`, using the same true line.
2. Take its training rows with `batch2.sample(frac=0.8, random_state=42)`.
3. Fit `line` and `quad` on those rows. Unpack the parameters into `a_line2, b_line2` and `a_quad2, b_quad2, c_quad2`.
4. Print the true log S at logP 5.0, which is $0.5 - 0.8 \cdot 5.0$.
5. For each model, print its prediction at logP 5.0 from batch 1 (Task 2's parameters) and from batch 2, both rounded to 2 decimals.

Which model's prediction moves more between the two batches? That model has the higher variance.


### Task 5 — Fix the model choice

A colleague wrote this to choose a model and report its performance:

```py
if r2_quad_test > r2_line_test:
    chosen = "quadratic"
    final_r2 = r2_quad_test
else:
    chosen = "straight line"
    final_r2 = r2_line_test
print("chosen model:", chosen, "test R2:", round(final_r2, 3))
```

This is wrong. The test set is choosing the model, so it has effectively become training data, and the reported score is optimistic.

Suppose the decision was made **before** any test score was seen, because log S was expected to fall on a straight line in logP. Rewrite the code so that:

- `chosen` is set to `"straight line"` directly, with no comparison
- the test set is scored **once**, for that model only, with `r_squared`
- the result prints in the same format as above

In track 6, cross-validation on the training data will replace the up-front decision.
